In [1]:
import altair as alt
alt.data_transformers.disable_max_rows()
import pandas as pd
import numpy as np

In [2]:
PRESETS = {
    "education-attainment": {
        "color": "#E00A0A", 
    },

    "education-attendance": {
        "color": "#E4936D",
    },
    "health-life-expectancy": {
        "color": "#095092",
    },
    "health-survival": {
        "color": "#0B0531",
    },

}

In [3]:
continent_map = {
    # Americas
    'Canada': 'North America', 'United States': 'North America', 'Mexico': 'Latin America & Caribbean',
    'Costa Rica': 'Latin America & Caribbean',
    'Chile': 'Latin America & Caribbean', 'Brazil': 'Latin America & Caribbean', 'Colombia': 'Latin America & Caribbean',

    # Europe - subdivided
    'Norway': 'Northern Europe', 'Sweden': 'Northern Europe', 'Finland': 'Northern Europe',
    'Denmark': 'Northern Europe', 'Iceland': 'Northern Europe', 'Estonia': 'Northern Europe',
    'Latvia': 'Northern Europe', 'Lithuania': 'Northern Europe',

    'United Kingdom': 'Western Europe', 'Ireland': 'Western Europe',
    'France': 'Western Europe', 'Belgium': 'Western Europe', 'Netherlands': 'Western Europe',
    'Luxembourg': 'Western Europe',

    'Spain': 'Southern Europe', 'Portugal': 'Southern Europe', 'Italy': 'Southern Europe',
    'Greece': 'Southern Europe', 'Cyprus': 'Southern Europe', 'Malta': 'Southern Europe',

    'Germany': 'Central Europe', 'Austria': 'Central Europe', 'Switzerland': 'Central Europe',
    'Poland': 'Central Europe', 'Czechia': 'Central Europe', 'Slovak Republic': 'Central Europe',
    'Hungary': 'Central Europe', 'Slovenia': 'Central Europe', 'Croatia': 'Central Europe',

    'Bulgaria': 'Eastern Europe & Balkans', 'Romania': 'Eastern Europe & Balkans',
    'Serbia': 'Eastern Europe & Balkans', 'Russia': 'Eastern Europe & Balkans',

    # Oceania
    'Australia': 'Oceania', 'New Zealand': 'Oceania',

    # Asia
    'Korea': 'East Asia',
    'Israel': 'Middle East',
    'Türkiye': 'Middle East'
}

In [4]:
continent_color = {
    # Americas
    'North America': "#9C27B0",      
    'Latin America & Caribbean': "#E67E22",  

    # Europe (different shades by sub-region)
    'Northern Europe': "#2A6DBB",     
    'Western Europe': "#4FC3F7",     
    'Southern Europe': "#00796B",    
    'Central Europe': "#0408F5",      
    'Eastern Europe & Balkans': "#388E3C",  

    # Oceania
    'Oceania': "#00ACC1",             

    # Asia
    'East Asia': "#E53935",               

    # Middle East 
    'Middle East': "#FDD835"          
}

In [5]:
# charts functions

def create_country_bar_chart(df, title_text="", axis_title="", color_t =""):
    """
    Creates a bar chart of education attainment by country.
    """
    chart = alt.Chart(df).mark_bar(color=color_t).encode(
        x=alt.X('Reference area:N', sort='-y', title='Country'),
        y=alt.Y('OBS_VALUE:Q', title=axis_title,
                axis=alt.Axis(grid=True, gridColor="#dbdbdb", gridOpacity=0.3)),
        tooltip=['Reference area', 'OBS_VALUE']
    ).properties(
        title=alt.TitleParams(
            text=title_text,
            anchor='start',
            offset=10,
            dx=20
        ),
        width=800,
        height=400
    ).resolve_legend(
        color='independent',
        shape='independent'
    )
    return chart

def create_regional_gap_chart(chart_df, country_order, scale_domain=None):
    """
    Creates a chart showing the gap between best and worst regions for each country.
    """
    gap_bars = alt.Chart(chart_df[chart_df['Type'].isin(['Worst Region', 'Best Region'])]).mark_rule(
        strokeWidth=8,
        stroke='lightgray',
        opacity=0.4
    ).encode(
        x=alt.X('Country:N', sort=country_order, title='Country', axis=alt.Axis(gridColor="#f3f1f1e1")),
        y=alt.Y('min(Value):Q', axis=alt.Axis(gridColor="#f3f1f1e1"),
            scale=alt.Scale(domain=scale_domain) if scale_domain else alt.Undefined),
        y2=alt.Y2('max(Value):Q'),
        tooltip=[
            alt.Tooltip('Country:N', title='Country'),
            alt.Tooltip('min(Value):Q', title='Worst Region Deviation'),
            alt.Tooltip('max(Value):Q', title='Best Region Deviation')
        ]
    )
    return gap_bars

def create_deviation_points_chart(chart_df, country_order, average_color, scale_domain=[0,100], y_title=""):
    """
    Creates a points chart for deviations from regional median.
    """
    points = alt.Chart(chart_df).mark_point(
        size=100,
        filled=True
    ).encode(
        x=alt.X('Country:N', sort=country_order, title='Country', axis=alt.Axis(gridColor="#f3f1f1e1")),
        y=alt.Y('Value:Q', title=y_title, scale=alt.Scale(domain=scale_domain)),
        shape=alt.Shape('Type:N',
            scale=alt.Scale(domain=['Best Region', 'National Average', 'Worst Region'],
                            range=['triangle-up', 'square', 'triangle-down']),
            legend=alt.Legend(orient='top'),
            title=''),
        color=alt.Color('Type:N',
            scale=alt.Scale(domain=['Best Region', 'National Average', 'Worst Region'],
                            range=['#3C3F3C', average_color, "#3C3F3C"]),
            legend=alt.Legend(orient='top'),
            title=''),
        tooltip=['Country', 'Type', 'Value', alt.Tooltip('WorstRegionName', title='Worst Region Name')]
    )
    text = alt.Chart(chart_df[chart_df['WorstRegionName'].notnull() & (chart_df['WorstRegionName'] != 'None') & (chart_df['WorstRegionName'] != '')]).mark_text(
        align='right',
        dy=0,
        dx=-10,
        fontSize=10,
        angle=270,      # makes text vertical
        color="#3C3F3C91"
    ).encode(
        x=alt.X('Country:N', sort=country_order),
        y='Value:Q',
        text='WorstRegionName:N'
    )
    return points + text

    """
    Creates a horizontal dashed line at y=0 for baseline.
    """
    zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
        strokeDash=[5, 5],
        stroke='black',
        strokeWidth=1.5,
        opacity=0.6
    ).encode(
        y=alt.Y('y:Q')
    )
    return zero_line

def create_source_text(width=800, height=20):
    """
    Creates a source text annotation for charts.
    """
    source_text = alt.Chart(pd.DataFrame({'source': ['Source: OECD']})).mark_text(
        align='left',
        fontSize=10,
        color='gray',
        dx=-400
    ).encode(
        text='source:N'
    ).properties(
        width=width,
        height=height
    )
    return source_text

def create_regional_variation_chart_no_norm(chart_df, country_order, average_color, chart_title= "", scale_domain=[0,100], y_title=""):
    """
    Combines gap bars, zero line, and deviation points into a single chart.
    """
    gap_bars = create_regional_gap_chart(chart_df, country_order, scale_domain)
    points = create_deviation_points_chart(chart_df, country_order, average_color, scale_domain, y_title)
    main_chart = (gap_bars + points).properties(
        title=alt.TitleParams(
            text=chart_title,
            anchor='start',
            offset=10,
            dx=20
        ),
        width=800,
        height=400
    ).resolve_legend(
        color='independent',
        shape='independent'
    )
    source_text = create_source_text()
    chart = alt.vconcat(
        main_chart,
        source_text,
        spacing=10
    ).resolve_scale(
        color='independent',
        shape='independent'
    )
    return chart

def scatter_gdp(merged_data, chart_title, y_title, chart_color, y_min, jump):

    scatter = alt.Chart(merged_data).mark_circle(size=150, color=chart_color).encode(
        x=alt.X(
            'log_2023:Q',
            title="GDP per Capita (2023 - log scale)",
            scale=alt.Scale(domain=[9, merged_data['log_2023'].max()+0.5]),
            axis=alt.Axis(
                format="d",  # This removes the .0
                grid=False, gridColor="#fffdfd", gridOpacity=0.3,
                values=list(range(9, int(np.ceil(merged_data['log_2023'].max())) + 1)),
                tickMinStep=1
            )
        ),
        y=alt.Y(
            'OBS_VALUE:Q', 
            title=y_title,
            scale=alt.Scale(domain=[y_min, merged_data['OBS_VALUE'].max()+0.5]),
            axis=alt.Axis(
                format="d",  # This removes the .0
                grid=False, gridColor="#fffdfd", gridOpacity=0.3,
                values=list(range(y_min, int(np.ceil(merged_data['OBS_VALUE'].max())) + 1, jump)),
            )
        ),
        tooltip=['Reference area', 'OBS_VALUE', '2023']
    )

    regression = alt.Chart(merged_data).transform_regression(
        'log_2023', 'OBS_VALUE'
    ).mark_line(color="#3F3E3EC3", strokeDash=[4, 2]).encode(
        x='log_2023:Q',
        y='OBS_VALUE:Q'
    )

    chart = (scatter + regression).properties(
        title=alt.TitleParams(
            text=chart_title,
            anchor='start',
            offset=30  # Increase this value for more space
    ),
    width=400,
    height=400
)
    return chart

def plot_gap_trend_facets(
    df,
    value_col="hci_gap",
    nregions_col="n_regions",
    country_col="Country",
    year_col="TIME_PERIOD",
    title="HCI Gap Selected Countries (2010–2024)",
    columns=6,
    color="#9C0505",
    width=130,
    height=70,
    country_order=None
):
    """
    Plot a faceted chart of gap trends for selected countries.
    """
    # Compute axis years for x-axis ticks
    _years = sorted(df[year_col].dropna().unique())
    if len(_years) >= 3:
        axis_years = [int(_years[0]), int(_years[len(_years)//2]), int(_years[-1])]
    else:
        axis_years = [int(y) for y in _years]

    x_axis = alt.X(
        f'{year_col}:O',
        title='Year',
        axis=alt.Axis(values=axis_years, labelAngle=0)
    )

    # Add per-country sort key (mean gap)
    base = (
        alt.Chart(df)
        .transform_joinaggregate(sort_key=f'mean({value_col})', groupby=[country_col])
    )

    line = (
        base.mark_line()
        .encode(
            x=x_axis,
            y=alt.Y(f'{value_col}:Q', title='Percentage points'),
            tooltip=[country_col, year_col, alt.Tooltip(f'{value_col}:Q', title='Gap'), nregions_col],
            color=alt.value(color)
        )
    )

    points = (
        base.mark_point(filled=True, size=30)
        .encode(
            x=x_axis,
            y=f'{value_col}:Q',
            tooltip=[country_col, year_col, alt.Tooltip(f'{value_col}:Q', title='Gap'), nregions_col],
            color=alt.value(color)
        )
    )

    panel = (line + points).properties(width=width, height=height)

    chart = panel.facet(
    facet=alt.Facet(
        f'{country_col}:N',
        sort=country_order if country_order is not None else alt.EncodingSortField(field='sort_key', order='descending'),
        title=""  # just a string
    ),
    columns=columns
).resolve_scale(y='shared').properties(
    title=alt.TitleParams(
        text=title,
        subtitle=".",
        subtitlePadding=20
    )
).configure_axis(grid=True)

    return chart

def plot_hci_change_chart(
    combined_points,
    country_order,
    continent_color,
    width=600,
    height_per_country=30,
    title="Human Capital Index (HCI), Circa 2010 - Circa 2024"
):
    """
    Plots HCI change from 2010 to 2024 for each country with lines and points.
    """
    lines = alt.Chart(combined_points).mark_line(
        color="#9B98982B",
        strokeWidth=5
    ).encode(
        y=alt.Y('Country:N', sort=country_order),
        x=alt.X('OBS_VALUE:Q', scale=alt.Scale(domain=[0.6, 0.9]), title='HCI'),
        detail='Country:N'
    )

    points = alt.Chart(combined_points).mark_point(
        filled=True,
        stroke='lightgray',
        strokeWidth=0.2
    ).encode(
        y=alt.Y('Country:N', sort=country_order, axis=alt.Axis(grid=True, gridColor="#e0e0e076")),
        x=alt.X('OBS_VALUE:Q', scale=alt.Scale(domain=[0.6, 0.9]), title='HCI', axis=alt.Axis(grid=True, gridColor="#e0e0e055")),
        shape=alt.Shape(
            'YearType:N',
            scale=alt.Scale(
                domain=['2010 National Average', '2024 National Average'],
                range=['square', 'triangle-right']
            ),
            legend=alt.Legend(title='Year', orient='bottom', columns=2)
        ),
        color=alt.condition(
            alt.datum.YearType == '2010 National Average',
            alt.value("#494a4bb7"),
            alt.Color('Continent:N',
                scale=alt.Scale(
                    domain=list(continent_color.keys()),
                    range=list(continent_color.values())
                ),
                legend=alt.Legend(title='World Region', orient='bottom', columns=3, rowPadding=5, offset=10)
            )
        ),
        size=alt.Size('YearType:N',
                      scale=alt.Scale(domain=['2010 National Average', '2024 National Average'],
                                      range=[60, 100])),
        tooltip=[alt.Tooltip('Country:N'), alt.Tooltip('OBS_VALUE:Q', format='.3f'), alt.Tooltip('YearType:N')]
    )

    source_text = create_source_text()

    chart = alt.layer(
        lines, points
    ).properties(
        width=width,
        height=height_per_country * len(country_order[:20]),
        title=alt.TitleParams(
            text=title,
            offset=50,
            dx=50,
            anchor='start'
        )
    )
    charts = alt.vconcat(
        chart,
        spacing=10
    ).configure_view(
        stroke=None
    )

    return charts

def add_contribution_shares(df, comp_cols=("LE_idx","Survival_idx","Attain_idx","Enroll_idx"), hci_col="HCI_composite"):
    """
    Allocate HCI_composite proportionally to components so contributions sum to HCI_composite.

    """
    eps = 1e-9
    dfc = df.copy()
    dfc["HCI_arith"] = dfc[list(comp_cols)].mean(axis=1)
    comp_sum = dfc[list(comp_cols)].sum(axis=1).replace(0, eps)
    for c in comp_cols:
        dfc[f"Contrib_{c}"] = (dfc[c] / comp_sum) * dfc[hci_col]
    return dfc

def plot_hci_stack(df, area_col="Reference area", top=10, weighted=True, colors=None, use_contributions=False, border_color="#FFFFFF3E", border_width=0.9, chart_title="Stacked contribution of HCI components"):
    """
    stacked HCI chart using the provided parameters.
    """
    comp_cols = ["LE_idx", "Survival_idx", "Attain_idx", "Enroll_idx"]

    # prepare plot dataframe and variable names
    if use_contributions:
        df_plot = add_contribution_shares(df, comp_cols=comp_cols)
        plot_vars = [f"Contrib_{c}" for c in comp_cols]
    else:
        df_plot = df.copy()
        plot_vars = comp_cols

    long = df_plot.melt(id_vars=[area_col, "HCI_composite"], value_vars=plot_vars, var_name="Component", value_name="Index")
    if use_contributions:
        long["Component"] = long["Component"].str.replace(r"^Contrib_", "", regex=True)
    if weighted and not use_contributions:
        long["Index"] = long["Index"] * 0.25

    # map internal component keys to friendly labels for legend/display ---
    label_map = {
        "LE_idx": "Life expectancy",
        "Survival_idx": "Survival rate",
        "Attain_idx": "Educational attainment",
        "Enroll_idx": "Enrollment rate"
    }
    long["Component"] = long["Component"].map(lambda k: label_map.get(k, k))

    # top countries selection
    top_countries = df.nlargest(top, "HCI_composite")[area_col].tolist()
    plot_df = long[long[area_col].isin(top_countries)]

    # color encoding (handles None, dict, or list) -- use mapped labels as domain
    if isinstance(colors, dict):
        mapped_domain = [label_map.get(k, k) for k in list(colors.keys())]
        color_enc = alt.Color("Component:N",
                              scale=alt.Scale(domain=mapped_domain, range=list(colors.values())),
                              legend=alt.Legend(title="", orient="top"))
    elif isinstance(colors, list):
        mapped_domain = [label_map[c] for c in comp_cols]
        color_enc = alt.Color("Component:N",
                              scale=alt.Scale(domain=mapped_domain, range=colors),
                              legend=alt.Legend(title="", orient="top"))
    else:
        color_enc = alt.Color("Component:N", legend=alt.Legend(title="", orient="top"))

    chart = (
        alt.Chart(plot_df)
        .mark_bar(stroke=border_color, strokeWidth=border_width)
        .encode(
            x=alt.X(f"{area_col}:N",
                    sort=alt.SortField(field="HCI_composite", order="descending"),
                    title="Country"),
            y=alt.Y("sum(Index):Q", stack="zero", title="HCI composite",
                    axis=alt.Axis(grid=True, gridColor="#dbdbdb", gridOpacity=0.3)),
            color=color_enc,
            tooltip=[area_col, "Component", "Index", alt.Tooltip("HCI_composite:Q", title="HCI")]
        )
        .properties(width=25 * top, height=400, title=alt.TitleParams(
            text=chart_title,
            anchor='start',
            offset=30  # Increase this value for more space
        ))
    ).configure_title(fontSize=16, anchor='start', dx=20)

    return chart

def plot_gap_trend_facets(
    df,
    value_col="hci_gap",
    nregions_col="n_regions",
    country_col="Country",
    year_col="TIME_PERIOD",
    title="HCI Gap Selected Countries (2010–2024)",
    columns=6,
    color="#9C0505",
    width=130,
    height=70,
    country_order=None
):
    """
    Plot  chart of gap trends for selected countries.
    """
    # Compute axis years for x-axis ticks
    _years = sorted(df[year_col].dropna().unique())
    if len(_years) >= 3:
        axis_years = [int(_years[0]), int(_years[len(_years)//2]), int(_years[-1])]
    else:
        axis_years = [int(y) for y in _years]

    x_axis = alt.X(
        f'{year_col}:O',
        title='Year',
        axis=alt.Axis(values=axis_years, labelAngle=0)
    )

    # Add per-country sort key (mean gap)
    base = (
        alt.Chart(df)
        .transform_joinaggregate(sort_key=f'mean({value_col})', groupby=[country_col])
    )

    line = (
        base.mark_line()
        .encode(
            x=x_axis,
            y=alt.Y(f'{value_col}:Q', title='Percentage points'),
            tooltip=[country_col, year_col, alt.Tooltip(f'{value_col}:Q', title='Gap'), nregions_col],
            color=alt.value(color)
        )
    )

    points = (
        base.mark_point(filled=True, size=30)
        .encode(
            x=x_axis,
            y=f'{value_col}:Q',
            tooltip=[country_col, year_col, alt.Tooltip(f'{value_col}:Q', title='Gap'), nregions_col],
            color=alt.value(color)
        )
    )

    panel = (line + points).properties(width=width, height=height)

    chart = panel.facet(
    facet=alt.Facet(
        f'{country_col}:N',
        sort=country_order if country_order is not None else alt.EncodingSortField(field='sort_key', order='descending'),
        title=""  # just a string
    ),
    columns=columns
).resolve_scale(y='shared').properties(
    title=alt.TitleParams(
        text=title,
        subtitle=".",
        subtitlePadding=20
    )
).configure_axis(grid=True)

    return chart

# Human capital index circa 2010 and 2024

In [6]:
combined = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\HCI_2010_2024.csv")

In [35]:
combined = combined.copy()
combined['Continent'] = combined['Country'].map(continent_map)

# Ensure CIRCA is string for filtering
combined['CIRCA'] = combined['CIRCA'].astype(str)

# Remove duplicate rows (keep one per country/year)
combined_unique = combined.drop_duplicates(subset=['Country', 'CIRCA'])

# Add a YearType column for plotting
combined_unique['YearType'] = combined_unique['CIRCA'].replace({'2010': '2010 National Average', '2024': '2024 National Average'})

# Filter for 2024 and 2010 only
combined_2024 = combined_unique[combined_unique['CIRCA'] == '2024']
combined_2010 = combined_unique[combined_unique['CIRCA'] == '2010']

# Combine for plotting
combined_points = pd.concat([combined_2010, combined_2024], ignore_index=True)

# Get country order by 2024 National Average (descending)
country_order = combined_2024.sort_values('OBS_VALUE', ascending=False)['Country'].tolist()

charts = plot_hci_change_chart(combined_points, country_order, continent_color)
charts.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\Shape_hci_change_2010_2024.svg")
charts

C:\Users\lopez\AppData\Local\Temp\ipykernel_34512\3314281177.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_unique['YearType'] = combined_unique['CIRCA'].replace({'2010': '2010 National Average', '2024': '2024 National Average'})


alt.VConcatChart(...)

# HEALTH INDICATORS

In [8]:
health_regional_data = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_regional_data.csv")
countries_with_additions = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_countries_complete.csv")

In [9]:
def filter_by_sex(df, sex='Total'):
    """Filter dataframe by sex."""
    return df[df['Sex'] == sex]

def get_regional_stats(df, min_regions=5):
    """
    Group by country and calculate min, max, count for OBS_VALUE.
    Also get Reference area for min and max OBS_VALUE.
    Filter by min_regions.
    """
    # Count regions per country
    stats = df.groupby('COUNTRY')['OBS_VALUE'].agg(['min', 'max', 'count']).reset_index()
    stats_filtered = stats[stats['count'] >= min_regions]

    # Get Reference area for min and max OBS_VALUE
    min_ref = df.loc[df.groupby('COUNTRY')['OBS_VALUE'].idxmin()][['COUNTRY', 'Reference area']].rename(columns={'Reference area': 'min_ref'})
    max_ref = df.loc[df.groupby('COUNTRY')['OBS_VALUE'].idxmax()][['COUNTRY', 'Reference area']].rename(columns={'Reference area': 'max_ref'})

    # Merge with stats_filtered
    stats_filtered = stats_filtered.merge(min_ref, on='COUNTRY', how='left')
    stats_filtered = stats_filtered.merge(max_ref, on='COUNTRY', how='left')

    return stats_filtered

def merge_country_names(stats_df, names_df):
    """Merge stats with country names."""
    return stats_df.merge(names_df[['COUNTRY', 'Country']].drop_duplicates(), on='COUNTRY')

def create_minmax_points(stats_df):
    """Create separate dataframes for min and max points and combine them."""
    min_points = stats_df[['Country', 'min', 'min_ref']].rename(columns={'min': 'OBS_VALUE', 'min_ref': 'Region name'})
    min_points['Type'] = 'Regional Min'
    max_points = stats_df[['Country', 'max', 'max_ref']].rename(columns={'max': 'OBS_VALUE', 'max_ref': 'Region name'})
    max_points['Type'] = 'Regional Max'
    minmax_points = pd.concat([min_points, max_points], ignore_index=True)
    return minmax_points

def filter_countries_with_regions(countries_df, valid_countries):
    """Filter country data to only include countries with enough regions."""
    return countries_df[countries_df['Reference area'].isin(valid_countries)]

def get_country_sort_order(countries_df):
    """Get country sorting order by national average."""
    return countries_df.sort_values('OBS_VALUE', ascending=False)['Reference area'].tolist()

def create_national_data(countries_df):
    """Prepare national average data for chart."""
    national_data = countries_df.copy()
    national_data['Type'] = 'National Average'
    national_data['Reference area'] = national_data['Reference area']
    return national_data[['Reference area', 'OBS_VALUE', 'Type']]

def combine_all_points(minmax_points, national_data):
    """Combine min/max points and national average data."""
    return pd.concat([minmax_points, national_data], ignore_index=True)

def merge_gdp_with_national_data(national_data, gdp_2023):
    """
    Merge GDP data with national health data using country names.
    Adds log_2023 as the natural log of the 2023 GDP value.
    Returns a DataFrame with GDP and health columns.
    """
    merged = national_data.merge(
        gdp_2023,
        left_on='Reference area',
        right_on='Country Name',
        how='left'
    )
    merged['log_2023'] = np.log(merged['2023'])
    return merged

# Data Preparation

regional_data_total = filter_by_sex(health_regional_data, sex='Total')
countries_with_additions_total = filter_by_sex(countries_with_additions, sex='Total')

regional_stats_filtered = get_regional_stats(regional_data_total, min_regions=5)
regional_stats_filtered = merge_country_names(regional_stats_filtered, regional_data_total)

minmax_points = create_minmax_points(regional_stats_filtered)
countries_with_5plus_regions = regional_stats_filtered['Country'].unique()
countries_filtered = filter_countries_with_regions(countries_with_additions_total, countries_with_5plus_regions)
country_sort_order = get_country_sort_order(countries_filtered)
national_data = create_national_data(countries_filtered)
all_points_data = combine_all_points(minmax_points, national_data)
all_points_data['Country'] = all_points_data['Country'].fillna(all_points_data['Reference area'])
all_points_data.rename(columns={'OBS_VALUE': 'Value'}, inplace=True)
all_points_data['Type'] = all_points_data['Type'].replace({
    'Regional Min': 'Worst Region',
    'Regional Max': 'Best Region'
})

all_points_data = all_points_data[~all_points_data['Country'].isin(['China (People’s Republic of)', 'Türkiye'])].copy()

all_points_data['WorstRegionName'] = all_points_data.apply(
    lambda row: row['Region name'] if row['Type'] == 'Worst Region' else None,
    axis=1
)
all_points_data['WorstRegionName'] = all_points_data['WorstRegionName'].astype(str)
country_order = national_data.sort_values('OBS_VALUE', ascending=False)['Reference area'].tolist()

In [10]:
chart_life_ine = create_regional_variation_chart_no_norm(
    all_points_data, 
    country_order, 
    average_color=PRESETS['health-life-expectancy']['color'], 
    chart_title="Regional Life Expectancy Variation by Country (Circa 2024)", 
    scale_domain=[50,90],
    y_title="Life Expectancy (years)")
chart_life_ine.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\shape_health_regional_life_expectancy_variation.svg")
chart_life_ine

alt.VConcatChart(...)

In [11]:
# GDP Scatter Plot
gdp_df = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\GDP.csv") 

gdp_2023 = gdp_df[['Country Name','Country Code',  '2023']]
merged_data = merge_gdp_with_national_data(national_data, gdp_2023) 

In [12]:
chart_gdp1 = scatter_gdp(merged_data, 'Life Expectancy vs GDP per Capita (2023 - log)', 'Life Expectancy (Years)',PRESETS['health-life-expectancy']['color'], 69, 5)
chart_gdp1.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_health_life_expectancy_vs_gdp.svg")
chart_gdp1

alt.LayerChart(...)

In [13]:
mortality_regional_data = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_childmort_regional_data.csv")
mortality_countries_with_additions = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_childmort_countries_complete.csv")

In [14]:
# transform to survival rate for better visualization
mortality_regional_data['OBS_VALUE'] = (1- (mortality_regional_data['OBS_VALUE'] / 1000))*100
#mortality_countries_with_additions['OBS_VALUE'] = (1- (mortality_countries_with_additions['OBS_VALUE'] / 1000))*100


In [15]:
regional_data_total = filter_by_sex(mortality_regional_data, sex='Total')
countries_with_additions_total = filter_by_sex(mortality_countries_with_additions, sex='Total')

regional_stats_filtered = get_regional_stats(regional_data_total, min_regions=5)
regional_stats_filtered = merge_country_names(regional_stats_filtered, regional_data_total)

minmax_points = create_minmax_points(regional_stats_filtered)
countries_with_5plus_regions = regional_stats_filtered['Country'].unique()
countries_filtered = filter_countries_with_regions(countries_with_additions_total, countries_with_5plus_regions)
country_sort_order = get_country_sort_order(countries_filtered)
national_data_survival = create_national_data(countries_filtered)
all_points_data = combine_all_points(minmax_points, national_data_survival)
all_points_data['Country'] = all_points_data['Country'].fillna(all_points_data['Reference area'])
all_points_data.rename(columns={'OBS_VALUE': 'Value'}, inplace=True)
all_points_data['Type'] = all_points_data['Type'].replace({
    'Regional Min': 'Worst Region',
    'Regional Max': 'Best Region'
})

all_points_data = all_points_data[~all_points_data['Country'].isin(['China (People’s Republic of)', 'Türkiye'])].copy()

all_points_data['WorstRegionName'] = all_points_data.apply(
    lambda row: row['Region name'] if row['Type'] == 'Worst Region' else None,
    axis=1
)
all_points_data['WorstRegionName'] = all_points_data['WorstRegionName'].astype(str)
country_order = national_data_survival.sort_values('OBS_VALUE', ascending=False)['Reference area'].tolist()

In [16]:
chart_survival = create_regional_variation_chart_no_norm(
    all_points_data, 
    country_order, 
    average_color=PRESETS['health-survival']['color'], 
    chart_title="Regional Survival Rate Variation by Country (Circa 2024)", 
    scale_domain=[90,100],
    y_title="Survival Rate (percentage)")
chart_survival.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\shape_health_regional_survival_rate_variation.svg")
chart_survival

alt.VConcatChart(...)

In [17]:
# GDP Scatter Plot
gdp_df = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\GDP.csv") 

gdp_2023 = gdp_df[['Country Name','Country Code',  '2023']]
merged_data = merge_gdp_with_national_data(national_data_survival, gdp_2023) 

In [18]:
chart_gdp2 = scatter_gdp(merged_data, 'Child survival rate vs GDP per Capita (2023 - log)', 'Child Survival Rate (%)',PRESETS['health-survival']['color'], 96, 2)
chart_gdp2.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_health_child_survival_vs_gdp.svg")
chart_gdp2

alt.LayerChart(...)

# Education indicators

In [19]:
attainment_regional_data = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\education_regional_data.csv")
attainment_countries_with_additions = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\education_countries_complete.csv")

In [20]:
regional_stats_filtered = get_regional_stats(attainment_regional_data, min_regions=5)
regional_stats_filtered = merge_country_names(regional_stats_filtered, attainment_regional_data)

minmax_points = create_minmax_points(regional_stats_filtered)
countries_with_5plus_regions = regional_stats_filtered['Country'].unique()
countries_filtered = filter_countries_with_regions(attainment_countries_with_additions, countries_with_5plus_regions)
country_sort_order = get_country_sort_order(countries_filtered)
attainment_national_data = create_national_data(countries_filtered)
all_points_data = combine_all_points(minmax_points, attainment_national_data)
all_points_data['Country'] = all_points_data['Country'].fillna(all_points_data['Reference area'])
all_points_data.rename(columns={'OBS_VALUE': 'Value'}, inplace=True)
all_points_data['Type'] = all_points_data['Type'].replace({
    'Regional Min': 'Worst Region',
    'Regional Max': 'Best Region'
})

all_points_data = all_points_data[~all_points_data['Country'].isin(['China (People’s Republic of)', 'Türkiye', 'Japan'])].copy()

all_points_data['WorstRegionName'] = all_points_data.apply(
    lambda row: row['Region name'] if row['Type'] == 'Worst Region' else None,
    axis=1
)
all_points_data['WorstRegionName'] = all_points_data['WorstRegionName'].astype(str)
all_points_data['WorstRegionName'] = all_points_data['WorstRegionName'].replace("Autonomous Region of the Azores", "Azores")
all_points_data['WorstRegionName'] = all_points_data['WorstRegionName'].replace("North Huetar", "Huetar")
all_points_data['WorstRegionName'] = all_points_data['WorstRegionName'].replace("Maranhão", "(1)")
country_order = attainment_national_data.sort_values('OBS_VALUE', ascending=False)['Reference area'].tolist()

In [21]:
chart_attainment = create_regional_variation_chart_no_norm(
    all_points_data, 
    country_order, 
    average_color=PRESETS['education-attainment']['color'], 
    chart_title="Regional Education Attainment Variation by Country (Circa 2024)", 
    scale_domain=[0,80],
    y_title="Education Attainment (percentage)")
chart_attainment.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\shape_education_regional_attainment_variation.svg")
chart_attainment

alt.VConcatChart(...)

In [22]:
attainment_merged_data = merge_gdp_with_national_data(attainment_national_data, gdp_2023)
chart_gdp3 = scatter_gdp(attainment_merged_data, 'Education Attainment vs GDP per Capita (2023 - log)', 'Education Attainment (%)', PRESETS["education-attainment"]["color"], 10,5)
chart_gdp3.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_education_attainment_vs_gdp.svg")
chart_gdp3

alt.LayerChart(...)

In [23]:
enrollment_regional_data = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\enrollment_regional_data.csv")
enrollment_with_additions = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\enrollment_countries_complete.csv")

In [24]:
regional_data_total = filter_by_sex(enrollment_regional_data, sex='Total')
countries_with_additions_total = filter_by_sex(enrollment_with_additions, sex='Total')

regional_stats_filtered = get_regional_stats(regional_data_total, min_regions=5)
regional_stats_filtered = merge_country_names(regional_stats_filtered, regional_data_total)

minmax_points = create_minmax_points(regional_stats_filtered)
countries_with_5plus_regions = regional_stats_filtered['Country'].unique()
countries_filtered = filter_countries_with_regions(countries_with_additions_total, countries_with_5plus_regions)
country_sort_order = get_country_sort_order(countries_filtered)
enrollment_national_data = create_national_data(countries_filtered)
all_points_data = combine_all_points(minmax_points, enrollment_national_data)

all_points_data['Country'] = all_points_data['Country'].fillna(all_points_data['Reference area'])
all_points_data.rename(columns={'OBS_VALUE': 'Value'}, inplace=True)
all_points_data['Type'] = all_points_data['Type'].replace({
    'Regional Min': 'Worst Region',
    'Regional Max': 'Best Region'
})

all_points_data = all_points_data[~all_points_data['Country'].isin(['China (People’s Republic of)', 'Türkiye', 'Hungary', 'Japan'])].copy()

all_points_data['WorstRegionName'] = all_points_data.apply(
    lambda row: row['Region name'] if row['Type'] == 'Worst Region' else None,
    axis=1
)
all_points_data['WorstRegionName'] = all_points_data['WorstRegionName'].astype(str)

country_order = enrollment_national_data.sort_values('OBS_VALUE', ascending=False)['Reference area'].tolist()
all_points_data.loc[(all_points_data['Country'] == 'Korea') & (all_points_data['Type'] == 'Best Region'), 'Value'] = 100

In [25]:
chart_enrollment = create_regional_variation_chart_no_norm(
    all_points_data, 
    country_order, 
    average_color=PRESETS['education-attendance']['color'], 
    chart_title="Education Attendance of 15-19 years old by Country and Region (Circa 2024)", 
    scale_domain=[0,100],
    y_title="Education Attendance (%)")
chart_enrollment.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\shape_education_regional_attendance_variation.svg")
chart_enrollment

alt.VConcatChart(...)

In [26]:
enrollment_merged_data = merge_gdp_with_national_data(enrollment_national_data, gdp_2023)
chart_gdp4 = scatter_gdp(enrollment_merged_data, 'School attendance of 15-19 years old vs GDP per Capita (2023 - log)', 'School Attendance (%)', PRESETS["education-attendance"]["color"], 60, 5)
chart_gdp4.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_education_school_attendance_vs_gdp.svg")
chart_gdp4

alt.LayerChart(...)

# building HCI index

In [27]:
def _norm_clip(series, lo=0.0, hi=1.0):
    s = pd.to_numeric(series, errors="coerce")
    return s.clip(lower=lo, upper=hi)

def _life_index(le_years: pd.Series, lo=20.0, hi=85.0):
    return _norm_clip((le_years - lo) / (hi - lo), 0.0, 1.0)

def _pct_index(pct: pd.Series):
    return _norm_clip(pct / 100.0, 0.0, 1.0)

def _gmean(a: pd.Series, b: pd.Series, c: pd.Series, d: pd.Series):
    return (a + b + c + d) * 0.25

def build_national_hci(Life: pd.DataFrame,
                       survival: pd.DataFrame,
                       attainment: pd.DataFrame,
                       enrollment: pd.DataFrame,
                       area_col: str = "Reference area",
                       value_col: str = "OBS_VALUE"):
    
    L = Life[[area_col, value_col]].rename(columns={value_col: "LE"})
    S = survival[[area_col, value_col]].rename(columns={value_col: "Survival"})
    A = attainment[[area_col, value_col]].rename(columns={value_col: "Attainment"})
    E = enrollment[[area_col, value_col]].rename(columns={value_col: "Enrollment"})
    df = L.merge(S, on=area_col, how="inner").merge(A, on=area_col, how="inner").merge(E, on=area_col, how="inner")
    df["LE_idx"] = _life_index(df["LE"])
    df["Survival_idx"] = _pct_index(df["Survival"])
   
    df["Attain_idx"] = _pct_index(df["Attainment"])
    df["Enroll_idx"] = _pct_index(df["Enrollment"])
   
    df["HCI_composite"] = _gmean(df["LE_idx"], df["Survival_idx"], df["Attain_idx"], df["Enroll_idx"])
    cols = [area_col, "LE", "Survival", "Attainment", "Enrollment",
            "LE_idx", "Survival_idx",  "Attain_idx", "Enroll_idx", "HCI_composite"]
    return df[cols].sort_values("HCI_composite", ascending=False).reset_index(drop=True)

In [28]:
Life = national_data
survival = national_data_survival
attainment = attainment_national_data
enrollment = enrollment_national_data
hci_national = build_national_hci(Life, survival, attainment, enrollment)

In [29]:
custom_colors = {
    "LE_idx": PRESETS['health-life-expectancy']['color'],       # blue
    "Survival_idx": PRESETS['health-survival']['color'],
    "Attain_idx": PRESETS["education-attainment"]["color"],
    "Enroll_idx": PRESETS["education-attendance"]["color"]    # red
}

plot_hci_stack(hci_national, top=30, weighted=False, colors=custom_colors, use_contributions=True, border_color="#FFFFFF00", border_width=0.9, chart_title="Stacked contribution of HCI components")

alt.Chart(...)

In [30]:
chart = plot_hci_stack(hci_national, top=30, weighted=False, colors=custom_colors, use_contributions=True)
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\stacked_hci_components_by_country.svg")

In [31]:
hci_all_years = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\regional_hci_all_years.csv")

In [32]:
hci_gap = (
    hci_all_years.groupby(['TIME_PERIOD', 'Country'])
    .agg(
        hci_gap=('HCI_composite', lambda x: x.max() - x.min()),
        n_regions=('Reference area', 'nunique')
    )
    .reset_index()
)
hci_gap = hci_gap[hci_gap['n_regions'] > 5]
hci_gap = hci_gap.rename(columns={'Country_x': 'Country'})

In [33]:

selected_countries = [
    "Colombia", "Mexico", "United States", "Austria", "Bulgaria" , "Czechia", "France", "Germany", "Greece", "Hungary",
    "Italy", "Netherlands", "Poland", "Romania", "Spain", "Sweden", "Switzerland", "United Kingdom"]


# Filter hci_gap for selected countries

filtered_hci_gap = hci_gap[hci_gap['Country'].isin(selected_countries)]
filtered_hci_gap["hci_gap"] = filtered_hci_gap["hci_gap"] * 100

latest_per_country = (
    filtered_hci_gap.sort_values('TIME_PERIOD')
    .groupby('Country')
    .tail(1)
    .sort_values('hci_gap', ascending=False)
)
country_order = latest_per_country['Country'].tolist()

chart2 = plot_gap_trend_facets(
    filtered_hci_gap,
    value_col="hci_gap",
    color="#353528",
    title="HCI Gap Selected Countries (2010–2024)",
    columns=6,
    width=150,
    height=70,
    country_order=country_order  # <-- pass the order here
)
chart2

C:\Users\lopez\AppData\Local\Temp\ipykernel_34512\1528782839.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_hci_gap["hci_gap"] = filtered_hci_gap["hci_gap"] * 100


alt.FacetChart(...)

In [34]:
chart2.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\facet_hci_gap_trends_selected_countries.svg")